<a href="https://colab.research.google.com/github/zhouning/alphaearth-training-system/blob/master/colab/train_loveda_crossdomain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LoveDA Cross-Domain PEFT Replication (paper12 Section 9.4)

Runs 5 PEFT methods × 2 directions (U→R, R→U) × 3 seeds = 30 runs on Colab Pro L4.

**Storage strategy (Drive-tight)**: only the result JSON (~50 KB) and the final merged JSON live on Drive. LoveDA imagery (~3 GB), Prithvi backbone (~450 MB), and per-run checkpoints (~1-3 GB) all live on Colab `/content/` and die when the session ends. Resume across sessions is driven by the result JSON's `done_keys` — already-finished `(method, seed)` rows are skipped on re-entry.

**Cost of session death**: at most one in-flight run (~50 min) is lost. LoveDA needs to be re-downloaded (~10 min, Zenodo). Plan for ~4-5 sessions to finish 30 runs.

Drive layout (assumed minimal):
- `MyDrive/Prithvi_100M.pt`        — backbone weights (already present from paper58, ~450 MB)
- `MyDrive/loveda/results/`        — result JSONs (the only thing this notebook writes to Drive, <100 KB total)

Reversal protocol (spec §5): if Houlsby ≈ Linear or LoRA ≥ +0.05 vs Linear, **stop and re-evaluate** before paper integration.

In [ ]:
# 1. Mount Drive (only used to (a) read Prithvi weights and (b) persist result JSON)
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs("/content/drive/MyDrive/loveda/results", exist_ok=True)

In [ ]:
# 2. GPU + Python sanity check + free disk
!nvidia-smi
!python --version
!df -h /content

In [ ]:
# 3. Clone the repo (public, no auth needed)
%cd /content
!rm -rf AlphaEarth-System
!git clone https://github.com/zhouning/alphaearth-training-system.git AlphaEarth-System
%cd AlphaEarth-System
!git log --oneline -5

In [ ]:
# 4. Install geoadapter (editable) + torchgeo + pyyaml
!pip install -q -e . torchgeo pyyaml

In [ ]:
# 5. Stage Prithvi backbone weights into the path geoadapter expects (Drive READ only — no write)
import shutil, os
os.makedirs("data/weights/prithvi", exist_ok=True)
DRIVE_WEIGHTS = "/content/drive/MyDrive/Prithvi_100M.pt"
DST = "data/weights/prithvi/Prithvi_100M.pt"
if not os.path.exists(DST):
    shutil.copy(DRIVE_WEIGHTS, DST)
print("Prithvi weights:", os.path.getsize(DST), "bytes")

In [ ]:
# 6. LoveDA cache lives entirely on /content (lost on session death — re-download is ~10 min)
import os
os.makedirs("data/weights/raw_data/loveda", exist_ok=True)
print("LoveDA root (ephemeral):", os.path.realpath("data/weights/raw_data/loveda"))

In [ ]:
# 7. Trigger torchgeo download into /content (first run downloads, subsequent runs in same session reuse)
from geoadapter.data.datasets import load_loveda
ds_smoke = load_loveda(root="data/weights/raw_data/loveda", domain="urban", split="train", max_samples=5)
print(f"LoveDA urban-train sample count: {len(ds_smoke)}")
img, mask = ds_smoke[0]
print(f"image shape={tuple(img.shape)}, mask shape={tuple(mask.shape)}, mask classes={set(mask.unique().tolist())}")

In [ ]:
# 8. PHASE 2 — U→R direction (15 runs: 5 methods × 3 seeds, 30 epochs each)
# Wall-clock ~12.5 h on L4. Result JSON on Drive drives the cross-session resume —
# completed (method, seed) rows are skipped on rerun. Per-run checkpoints (~/runs)
# live on /content and survive only within one session.
!mkdir -p /content/loveda_runs/u2r
!python -m geoadapter.bench.run_benchmark \
    --config geoadapter/bench/configs/loveda_lulc_u2r.yaml \
    --output /content/drive/MyDrive/loveda/results/loveda_lulc_seg_u2r.json \
    --checkpoint-dir /content/loveda_runs/u2r \
    --checkpoint-every 5

In [ ]:
# 9. Verify all 15 U→R rows are present before moving on
import json
rows = json.loads(open("/content/drive/MyDrive/loveda/results/loveda_lulc_seg_u2r.json").read())
assert len(rows) == 15, f"expected 15 rows, got {len(rows)}"
print("U→R completed pairs:", sorted({(r['method'], r['seed']) for r in rows}))

In [ ]:
# 10. PHASE 2 — R→U direction (15 runs)
!mkdir -p /content/loveda_runs/r2u
!python -m geoadapter.bench.run_benchmark \
    --config geoadapter/bench/configs/loveda_lulc_r2u.yaml \
    --output /content/drive/MyDrive/loveda/results/loveda_lulc_seg_r2u.json \
    --checkpoint-dir /content/loveda_runs/r2u \
    --checkpoint-every 5

In [ ]:
# 11. Verify all 15 R→U rows + merge into the canonical paper artifact
import json
rows = json.loads(open("/content/drive/MyDrive/loveda/results/loveda_lulc_seg_r2u.json").read())
assert len(rows) == 15, f"expected 15 rows, got {len(rows)}"
print("R→U completed pairs:", sorted({(r['method'], r['seed']) for r in rows}))

!python -m geoadapter.bench.run_loveda_crossdomain \
    --u2r-config geoadapter/bench/configs/loveda_lulc_u2r.yaml \
    --r2u-config geoadapter/bench/configs/loveda_lulc_r2u.yaml \
    --output /content/drive/MyDrive/loveda/results/loveda_lulc_seg.json \
    --skip-runs

import os, shutil
os.makedirs("results/loveda", exist_ok=True)
shutil.copy("/content/drive/MyDrive/loveda/results/loveda_lulc_seg.json",
            "results/loveda/loveda_lulc_seg.json")
print("Local copy ready at results/loveda/loveda_lulc_seg.json")

In [ ]:
# 12. Summary table for LaTeX + reversal-protocol gate (spec §5)
import json, statistics
rows = json.loads(open("results/loveda/loveda_lulc_seg.json").read())
agg = {}
for r in rows:
    agg.setdefault((r['direction'], r['method']), []).append(r['mIoU'])
print(f"{'direction':<8} {'method':<14} {'mean':>8} {'std':>8} {'n':>3}")
for (d, m), v in sorted(agg.items()):
    print(f"{d:<8} {m:<14} {statistics.mean(v):>8.4f} {statistics.stdev(v) if len(v)>1 else 0:>8.4f} {len(v):>3}")

def mean_for(d, m): return statistics.mean(agg[(d, m)])
u2r_houlsby_gain = mean_for('U->R', 'houlsby')      - mean_for('U->R', 'linear_probe')
r2u_houlsby_gain = mean_for('R->U', 'houlsby')      - mean_for('R->U', 'linear_probe')
u2r_lora_gap    = abs(mean_for('U->R', 'lora_r8')   - mean_for('U->R', 'linear_probe'))
r2u_lora_gap    = abs(mean_for('R->U', 'lora_r8')   - mean_for('R->U', 'linear_probe'))
print()
print(f"Houlsby − Linear   U→R: {u2r_houlsby_gain:+.4f}   R→U: {r2u_houlsby_gain:+.4f}")
print(f"|LoRA − Linear|    U→R: {u2r_lora_gap:.4f}   R→U: {r2u_lora_gap:.4f}")
houlsby_ok = (u2r_houlsby_gain >= 0.05) or (r2u_houlsby_gain >= 0.05)
lora_ok    = (u2r_lora_gap < 0.05) and (r2u_lora_gap < 0.05)
if houlsby_ok and lora_ok:
    print("\nREVERSAL PROTOCOL: PASS — ranking reproduces, proceed to Phase 3 paper integration.")
else:
    print("\nREVERSAL PROTOCOL: FAIL — STOP. Do not write Section 9.4. Re-evaluate per spec §5.")

## Step 8.5 — Capacity-threshold diagnostic (post hoc)

Phase 2's reversal-protocol gate (cell 12) passed, but absolute mIoU for `linear_probe / bitfit / lora_r8 / geoadapter` clusters at ≈0.086 (U→R) / ≈0.095 (R→U) — within noise of the all-class-0 majority-class baseline (LoveDA bg ≈ 60% of pixels → mIoU ≈ 0.086).

We need to know whether small PEFT methods are:

- **(A) collapsing to majority-class prediction** → the *capacity-threshold* finding for Section 9.4; or
- **(B) learning weak signal that mIoU under-reports** → softens the Section 9.4 claim.

Cells 13-14 run a single-seed sweep over `lora_r8 / lora_r16` with **lr_peft=5e-4 (5×) and epochs=60 (2×)**. The eval print line now shows per-class IoU and dominant-prediction share. If we still see `[COLLAPSE]` and per-class IoU ≈ `[0.5, 0, 0, 0, 0, 0, 0]`, finding A is confirmed and Section 9.4 narrative is locked.

In [ ]:
# 13. Diagnostic — re-clone for the Step 8.5 patch (eval per-class IoU + diag config)
%cd /content
!cd AlphaEarth-System && git pull --ff-only
!cd AlphaEarth-System && git log --oneline -3

In [ ]:
# 14. Run the U→R LoRA-rank diagnostic sweep (2 methods × 1 seed × 60 epochs ≈ 1.5 h on L4)
# Result lands on /content (NOT Drive) — diagnostic only, not a paper artifact.
%cd /content/AlphaEarth-System
!mkdir -p /content/loveda_runs/u2r_diag /content/diag_results
!python -m geoadapter.bench.run_benchmark \
    --config geoadapter/bench/configs/loveda_lulc_u2r_diag.yaml \
    --output /content/diag_results/loveda_u2r_diag.json \
    --checkpoint-dir /content/loveda_runs/u2r_diag \
    --checkpoint-every 10

In [ ]:
# 15. Diagnostic verdict — collapse vs weak-signal
import json
diag = json.loads(open("/content/diag_results/loveda_u2r_diag.json").read())

print(f"{'method':<14} {'mIoU':>8} {'class0':>8} {'class1+':>10} {'dom_pred':>8} {'verdict'}")
for r in diag:
    pcs = r.get("per_class_iou") or []
    pred_total = sum(r.get("pred_pixel_count", []) or [0]) or 1
    pred_share = [p / pred_total for p in r.get("pred_pixel_count", [])]
    top_pred = max(range(len(pred_share)), key=lambda i: pred_share[i]) if pred_share else -1
    top_share = pred_share[top_pred] if pred_share else 0.0
    cls0 = pcs[0] if pcs else None
    cls_rest = [v for v in pcs[1:] if v is not None]
    rest_max = max(cls_rest) if cls_rest else 0.0
    if top_share > 0.95:
        verdict = "COLLAPSE → finding (A) confirmed"
    elif rest_max > 0.10:
        verdict = "WEAK-SIGNAL → finding (B), revisit narrative"
    else:
        verdict = "AMBIGUOUS → consider lr/epochs sweep"
    print(f"{r['method']:<14} {r['mIoU']:>8.4f} "
          f"{(cls0 if cls0 is not None else float('nan')):>8.4f} "
          f"{rest_max:>10.4f} {top_pred}({top_share:.0%}) {verdict}")

print()
# Compare to Phase 2 lora_r8 (mIoU ≈ 0.0855 with lr_peft=1e-4, 30 epochs)
phase2_baseline = 0.0855
gain = max(r["mIoU"] for r in diag) - phase2_baseline
print(f"Best diag mIoU vs Phase 2 lora_r8 baseline: {gain:+.4f}")
if gain < 0.02:
    print("→ Hyperparams ruled out. Section 9.4 capacity-threshold narrative LOCKED.")
elif gain < 0.05:
    print("→ Borderline gain. Mention as caveat in Section 9.4 but ranking still holds.")
else:
    print("→ Substantial hyperparam gain. RECONSIDER — small PEFT may have been undertrained.")